# Tutorial ROSS - Part 6: Squeeze Film Dampers (SFD)
This tutorial demonstrates how to use the custom SqueezeFilmDamper class to model non-rotating oil film damping elements in rotordynamic systems using the ROSS library.

# Section 1: Introduction to Squeeze Film Dampers
A Squeeze Film Damper (SFD) is a support element that provides additional damping to rotors to reduce vibration amplitudes and improve stability. Unlike a standard hydrodynamic bearing, the inner ring of an SFD does not rotate; it only precesses, "squeezing" the oil film against the housing.

# 1.1 Results
In this tutorial, we use the Classical Short-Bearing Theory to compute:
- $c_o$: Damping coefficient.$k_o$. 
- Stiffness coefficient (added mass/stiffness effect).
- $P_{max}$: Maximum pressure in the oil film.
- $\theta_m$: Pressure angle.

# 1.2 Mathematical Backgorund : Reynold's Equation.
The behavior of the Squeeze Film Damper is governed by the Reynolds Equation for thin film lubrication. For a non-rotating journal (SFD case), where the velocity is purely translational ($\dot{e}$ and $e\dot{\phi}$), the simplified Reynolds Equation in cylindrical coordinates is:
$$\frac{1}{R^2} \frac{\partial}{\partial \theta} \left( h^3 \frac{\partial p}{\partial \theta} \right) + \frac{\partial}{\partial z} \left( h^3 \frac{\partial p}{\partial z} \right) = -12\mu ( \dot{e} \cos \theta + e\dot{\phi} \sin \theta )$$
- Where:$h = C(1 + \epsilon \cos \theta)$ is the local film thickness.
- $\mu$ is the dynamic viscosity.
- $R$ is the journal radius.

The Short Bearing ApproximationFor dampers where the length-to-diameter ratio is small ($L/D < 0.5$), we assume the pressure gradient in the circumferential direction is much smaller than in the axial direction ($\frac{\partial p}{\partial \theta} \ll \frac{\partial p}{\partial z}$). This allows us to drop the first term, leading to the analytical solution for pressure $p(\theta, z)$ used in your code.

# Section 2: Comparasion of the three Geometric cases
The damping ($c_o$) and stiffness ($k_o$) coefficients change depending on how the fluid is contained and how the oil is supplied.
- Case A: End Seals (No Groove)
Physical setup: The damper has seals (like O-rings) at the ends to prevent oil from leaking axially.
* Mechanism: Because the oil is "trapped," the pressure builds up significantly higher.
* Equation Logic: The pressure distribution is assumed to be more uniform axially, leading to the highest damping values.



- Case B: Central Groove (No End Seals)
Physical setup: A circumferential groove is machined in the middle of the bearing to supply oil.
* Mechanism: The groove effectively splits the damper into two shorter dampers of length $L/2$.
* Equation Logic: Since damping is proportional to $L^3$ in short bearing theory, splitting the length significantly reduces the total damping ($(\frac{L}{2})^3 + (\frac{L}{2})^3 = \frac{1}{4} L^3$).



- Case C: Central Groove + End Seals
Physical setup: A combination of a supply groove and axial seals.
* Mechanism: This setup balances the constant oil supply from the groove with the pressure-retention capabilities of the seals.
* Equation Logic: It uses the full length $L$ in the pressure equations but accounts for the boundary conditions imposed by the seals.

# Section 3: Cavitation

* Non-Cavitated ($2\pi$): The oil film is continuous. The negative pressure in the divergent zone is equal to the positive pressure in the convergent zone. Result: High Damping, Zero Stiffness.
* Cavitated ($\pi$): The oil "boils" or releases air when pressure drops below vapor/ambient pressure. We ignore the negative pressure region. Result: Reduced Damping, but generates "Cross-coupled" Stiffness ($k_o$).

# Section 4: Environmente Setup


In [ ]:
import ross as rs
import numpy as np
import matplotlib.pyplot as plt

# Using Pint for unit management
Q_ = rs.units.Q_

# Section 5: Defining SFD Parameters
| Parameter | Description | Suggested Unit |
| :--- | :--- | :--- |
| `n` | Node location | Int |
| `axial_length` | Damper length ($L$) | inches or mm |
| `journal_radius` | Journal radius ($R$) | inches or mm |
| `radial_clearance` | Radial clearance ($C$) | inches or mm |
| `eccentricity_ratio` | $\epsilon = e/C$ | Dimensionless (0.0 to 1.0) |
| `lubricant` | Lubricant type | String (e.g., 'ISOVG32') |